# Guardrailer Embedding Model Evaluation: bge-large-en-v1.5
## Centroid Similarity, Linear Probes, and AUC-ROC on 10,000 Samples

**Runtime:** Enable GPU (T4) in Kaggle Settings → Accelerator → GPU T4 x2
**Checkpointing:** All progress auto-saves to /kaggle/working/checkpoints/
**Resume:** If interrupted, re-run all cells — notebook resumes from last checkpoint

In [ ]:
!pip install -q sentence-transformers scikit-learn pandas numpy matplotlib seaborn

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU Memory: {props.total_memory / 1e9:.1f} GB")
    print(f"Compute Capability: {props.major}.{props.minor}")

CUDA_WORKS = False
if torch.cuda.is_available():
    try:
        _t = torch.randn(10).cuda()
        _t = _t * 2
        del _t
        torch.cuda.empty_cache()
        CUDA_WORKS = True
        print("CUDA kernel test: PASSED")
    except Exception as e:
        print(f"CUDA kernel test: FAILED ({e})")
else:
    print("No GPU detected.")

DEVICE = "cuda" if CUDA_WORKS else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix, roc_curve
from sentence_transformers import SentenceTransformer

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 150

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
OUTPUT_DIR = "/kaggle/working"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

class CheckpointManager:
    def __init__(self, checkpoint_dir=CHECKPOINT_DIR):
        self.checkpoint_dir = checkpoint_dir
        self.manifest_path = os.path.join(checkpoint_dir, "manifest.json")
        self.manifest = self._load_manifest()

    @staticmethod
    def _safe_key(name):
        return name.replace("/", "_")

    def _load_manifest(self):
        if os.path.exists(self.manifest_path):
            with open(self.manifest_path, "r") as f:
                return json.load(f)
        return {"completed_models": [], "saved_embeddings": {}}

    def _save_manifest(self):
        with open(self.manifest_path, "w") as f:
            json.dump(self.manifest, f, indent=2)

    def is_model_done(self, model_name):
        return model_name in self.manifest["completed_models"]

    def save_embeddings(self, model_name, split, embeddings, labels):
        key = f"{self._safe_key(model_name)}_{split}"
        emb_path = os.path.join(self.checkpoint_dir, f"{key}_embeddings.npy")
        label_path = os.path.join(self.checkpoint_dir, f"{key}_labels.npy")
        np.save(emb_path, embeddings)
        np.save(label_path, labels)
        self.manifest["saved_embeddings"][key] = {
            "path": emb_path,
            "label_path": label_path,
            "shape": list(embeddings.shape),
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        }
        self._save_manifest()
        print(f"  Checkpoint saved: {key} ({embeddings.shape[0]} samples, {embeddings.shape[1]} dims)")

    def load_embeddings(self, model_name, split):
        key = f"{self._safe_key(model_name)}_{split}"
        if key in self.manifest["saved_embeddings"]:
            info = self.manifest["saved_embeddings"][key]
            if os.path.exists(info["path"]) and os.path.exists(info["label_path"]):
                embeddings = np.load(info["path"])
                labels = np.load(info["label_path"])
                print(f"  Loaded checkpoint: {key} ({embeddings.shape[0]} samples)")
                return embeddings, labels
        return None, None

    def save_proba(self, model_name, proba, y_test):
        key = self._safe_key(model_name)
        path = os.path.join(self.checkpoint_dir, f"{key}_proba.npy")
        label_path = os.path.join(self.checkpoint_dir, f"{key}_ytest.npy")
        np.save(path, proba)
        np.save(label_path, y_test)

    def load_proba(self, model_name):
        key = self._safe_key(model_name)
        path = os.path.join(self.checkpoint_dir, f"{key}_proba.npy")
        label_path = os.path.join(self.checkpoint_dir, f"{key}_ytest.npy")
        if os.path.exists(path) and os.path.exists(label_path):
            proba = np.load(path)
            y_test = np.load(label_path)
            print(f"  Loaded cached proba: {key}")
            return proba, y_test, None
        return None

    def mark_model_done(self, model_name, results):
        self.manifest["completed_models"].append(model_name)
        self.manifest[f"results_{model_name}"] = results
        self._save_manifest()
        print(f"  Model marked complete: {model_name}")

    def get_results(self, model_name):
        return self.manifest.get(f"results_{model_name}", None)

    def get_status(self):
        print(f"\n{'='*50}")
        print(f"CHECKPOINT STATUS")
        print(f"{'='*50}")
        print(f"Completed models: {self.manifest['completed_models']}")
        print(f"Saved embeddings: {list(self.manifest['saved_embeddings'].keys())}")
        print(f"{'='*50}\n")

ckpt = CheckpointManager()
ckpt.get_status()

In [ ]:
DATA_PATH = "/kaggle/input/datasets/prashannadeveloper/guardrailer-dataset-v1/guardrailer_dataset_v1.parquet"

df = pd.read_parquet(DATA_PATH)
df["text"] = df["prompt_text"]
df["label"] = df["is_malicious"].astype(int)

print(f"Total samples: {len(df)}")
print(f"\nLabel distribution:")
print(df["label"].value_counts())
print(f"\nLabel distribution (normalized):")
print(df["label"].value_counts(normalize=True))

print("\n--- Sample malicious text ---")
print(df[df["label"] == 1]["text"].iloc[0][:200])
print("\n--- Sample benign text ---")
print(df[df["label"] == 0]["text"].iloc[0][:200])

In [ ]:
# Stratified 10,000 sample subset
subset, _ = train_test_split(
    df, train_size=10000, random_state=42, stratify=df["label"]
)

print(f"Subset: {len(subset)} samples")
print(f"Class distribution: {subset['label'].value_counts(normalize=True).to_dict()}")

# Train/test split (80/20)
sub_train, sub_test = train_test_split(
    subset, test_size=0.20, random_state=42, stratify=subset["label"]
)

X_train = sub_train["text"].astype(str).tolist()
y_train = sub_train["label"].values
X_test = sub_test["text"].astype(str).tolist()
y_test = sub_test["label"].values

print(f"Train: {len(X_train)}  Test: {len(X_test)}")
print(f"Train class dist: safe={sum(y_train==0)}, malicious={sum(y_train==1)}")
print(f"Test class dist:  safe={sum(y_test==0)}, malicious={sum(y_test==1)}")

In [ ]:
MODEL_NAME = "BAAI/bge-large-en-v1.5"

# Check checkpoint
if ckpt.is_model_done(MODEL_NAME):
    print(f"{MODEL_NAME} already completed. Loading from checkpoint.")
    cached = ckpt.load_proba(MODEL_NAME)
    saved = ckpt.get_results(MODEL_NAME)
    if cached is not None:
        proba, y_test_loaded, _ = cached
        y_test = y_test_loaded
    train_emb, train_labels = ckpt.load_embeddings(MODEL_NAME, "train")
    test_emb, test_labels = ckpt.load_embeddings(MODEL_NAME, "test")
else:
    print(f"Encoding with {MODEL_NAME}...")
    model = SentenceTransformer(MODEL_NAME)
    print(f"  Model loaded. Device: {DEVICE}")

    start = time.time()
    print(f"  Encoding train set ({len(X_train)} samples)...")
    train_emb = model.encode(X_train, batch_size=64, show_progress_bar=True,
                             normalize_embeddings=True, device=DEVICE)
    train_emb = np.array(train_emb)
    train_labels = np.array(y_train)
    ckpt.save_embeddings(MODEL_NAME, "train", train_emb, train_labels)

    print(f"  Encoding test set ({len(X_test)} samples)...")
    test_emb = model.encode(X_test, batch_size=64, show_progress_bar=True,
                            normalize_embeddings=True, device=DEVICE)
    test_emb = np.array(test_emb)
    test_labels = np.array(y_test)
    ckpt.save_embeddings(MODEL_NAME, "test", test_emb, test_labels)

    encode_time = time.time() - start
    print(f"  Encoding time: {encode_time:.1f}s")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Centroid similarity
    centroid_safe = train_emb[train_labels == 0].mean(axis=0)
    centroid_malicious = train_emb[train_labels == 1].mean(axis=0)
    centroid_similarity = np.dot(centroid_safe, centroid_malicious)

    test_safe_emb = test_emb[test_labels == 0]
    test_mal_emb = test_emb[test_labels == 1]
    test_centroid_sim = np.dot(test_safe_emb.mean(axis=0), test_mal_emb.mean(axis=0))

    safe_to_centroid = np.mean([np.dot(e, centroid_safe) for e in test_safe_emb])
    mal_to_centroid = np.mean([np.dot(e, centroid_malicious) for e in test_mal_emb])

    print(f"\n  Centroid Similarity (train): {centroid_similarity:.6f}")
    print(f"  Centroid Similarity (test):  {test_centroid_sim:.6f}")

    # Linear probe
    print("  Training linear probe...")
    probe = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    probe.fit(train_emb, train_labels)
    preds = probe.predict(test_emb)
    proba = probe.predict_proba(test_emb)[:, 1]

    acc = accuracy_score(test_labels, preds)
    f1 = f1_score(test_labels, preds)
    prec = precision_score(test_labels, preds)
    rec = recall_score(test_labels, preds)
    auc = roc_auc_score(test_labels, proba)
    cm = confusion_matrix(test_labels, preds)

    print(f"\n  Linear Probe Results:")
    print(f"    Accuracy:  {acc:.4f}")
    print(f"    F1-Score:  {f1:.4f}")
    print(f"    Precision: {prec:.4f}")
    print(f"    Recall:    {rec:.4f}")
    print(f"    AUC-ROC:   {auc:.4f}")
    print(f"    Confusion Matrix:")
    print(f"      TN={cm[0,0]:,}  FP={cm[0,1]:,}")
    print(f"      FN={cm[1,0]:,}  TP={cm[1,1]:,}")

    # Save proba for plotting
    ckpt.save_proba(MODEL_NAME, proba, test_labels)

    saved = {
        "model_name": MODEL_NAME,
        "centroid_similarity_train": float(centroid_similarity),
        "centroid_similarity_test": float(test_centroid_sim),
        "safe_to_centroid": float(safe_to_centroid),
        "malicious_to_centroid": float(mal_to_centroid),
        "accuracy": float(acc),
        "f1": float(f1),
        "precision": float(prec),
        "recall": float(rec),
        "auc_roc": float(auc),
        "confusion_matrix": cm.tolist(),
        "encode_time_sec": float(encode_time),
    }
    ckpt.mark_model_done(MODEL_NAME, saved)

print(f"\n{'='*50}")
print(json.dumps(saved, indent=2))

In [ ]:
# === Visualization ===
output_dir = "/kaggle/working/"
short_name = MODEL_NAME.split("/")[-1]

# Figure 1: Centroid Similarity
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(["Train", "Test"],
              [saved["centroid_similarity_train"], saved["centroid_similarity_test"]],
              color=["#2196F3", "#FF5722"], alpha=0.8)
ax.axhline(y=0.85, color="green", linestyle="--", alpha=0.5, label="Threshold (0.85)")
ax.set_ylabel("Cosine Similarity")
ax.set_title(f"Inter-Class Centroid Similarity: {short_name}")
ax.set_ylim(0.5, 1.05)
ax.legend()
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=11)
plt.tight_layout()
plt.savefig(output_dir + "fig_centroid_similarity.png", dpi=150, bbox_inches="tight")
plt.show()

# Figure 2: Metrics Bar Chart
fig, ax = plt.subplots(figsize=(8, 5))
metrics = ["accuracy", "f1", "precision", "recall", "auc_roc"]
labels = ["Accuracy", "F1-Score", "Precision", "Recall", "AUC-ROC"]
colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#F44336"]
values = [saved[m] for m in metrics]
bars = ax.bar(labels, values, color=colors, alpha=0.8)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f"{val:.4f}", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("Score")
ax.set_title(f"Linear Probe Performance: {short_name}")
ax.set_ylim(0.4, 1.05)
plt.tight_layout()
plt.savefig(output_dir + "fig_linear_probe_performance.png", dpi=150, bbox_inches="tight")
plt.show()

# Figure 3: ROC Curve
fig, ax = plt.subplots(figsize=(8, 6))
fpr, tpr, _ = roc_curve(y_test, proba)
ax.plot(fpr, tpr, color="#2196F3", linewidth=2,
        label=f'{short_name} (AUC={saved["auc_roc"]:.3f})')
ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random (AUC=0.500)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(f"ROC Curve: {short_name}")
ax.legend(loc="lower right")
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
plt.tight_layout()
plt.savefig(output_dir + "fig_roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()

# Figure 4: Score Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
safe_scores = proba[y_test == 0]
mal_scores = proba[y_test == 1]
axes[0].hist(safe_scores, bins=50, alpha=0.7, color="#4CAF50", label="Safe", density=True)
axes[0].hist(mal_scores, bins=50, alpha=0.7, color="#F44336", label="Malicious", density=True)
axes[0].set_xlabel("Predicted Probability (Malicious)")
axes[0].set_ylabel("Density")
axes[0].set_title(f"Score Distribution: {short_name}")
axes[0].legend()
margins_dist = mal_scores - safe_scores.mean()
axes[1].hist(margins_dist, bins=50, color="#FF9800", alpha=0.7, edgecolor="black")
axes[1].axvline(x=0, color="red", linestyle="--", linewidth=2, label="Decision boundary")
axes[1].set_xlabel("Score Margin (relative to safe centroid)")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Margin Distribution: {short_name}")
axes[1].legend()
plt.tight_layout()
plt.savefig(output_dir + "fig_score_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print("All figures saved.")

In [ ]:
# Save summary
output_dir = "/kaggle/working/"
with open(output_dir + "bge_large_results.json", "w") as f:
    json.dump(saved, f, indent=2)
print("Saved: bge_large_results.json")

results_df = pd.DataFrame([saved])
results_df.to_csv(output_dir + "bge_large_results.csv", index=False)
print("Saved: bge_large_results.csv")

print(f"\n=== RESULTS: {MODEL_NAME} ===")
print(results_df[["model_name", "centroid_similarity_test", "accuracy", "f1", "auc_roc"]].to_string(index=False))

## Expected Outputs

### Files
- `bge_large_results.json` — Full metrics
- `bge_large_results.csv` — Tabular format
- `fig_centroid_similarity.png` — Centroid similarity bar chart
- `fig_linear_probe_performance.png` — Multi-metric bar chart
- `fig_roc_curve.png` — ROC curve
- `fig_score_distribution.png` — Score distribution + margin analysis

### Collapse Criteria
The model is classified as "collapsed" when ALL of:
1. Inter-class centroid similarity > 0.95
2. Linear probe AUC-ROC < 0.55
3. Linear probe accuracy < majority-class baseline